In [1]:
!pip -q install typer[all] pandera scikit-learn pandas pyarrow polars

# dev install after you push to GitHub (example)
# !pip -q install git+https://github.com/yourname/oncopanel.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.9/292.9 kB 12.1 MB/s eta 0:00:00


In [2]:
%%bash
set -euo pipefail

BASE=/content/oncopanel
mkdir -p $BASE/{dataloaders,validators,calibrators,metrics,bootstrap,dca,aggregate,receipts,configs,tests}

# ----- package inits -----
for d in "" dataloaders validators calibrators metrics bootstrap dca aggregate receipts; do
  printf "__all__ = []\n" > $BASE/${d:+$d/}__init__.py
done

# ----- pyproject -----
cat > $BASE/pyproject.toml <<'TOML'
[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "oncopanel"
version = "0.1.0"
description = "Panel-scale context vs codon evaluation (calibration, DCA, bootstrap)"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
  "numpy",
  "pandas",
  "pyarrow",
  "polars>=0.20.0",
  "scikit-learn",
  "typer>=0.12",
  "pandera>=0.18",
]

[project.scripts]
oncopanel = "oncopanel.cli:main"
TOML

# ----- README -----
cat > $BASE/README.md <<'MD'
# oncopanel
Boring, fast, audit-proof panel evaluation at scale (Colab-friendly).

## Quickstart (Colab)
```bash
!pip install -e /content/oncopanel
!oncopanel --help


bash: line 45: warning: here-document at line 38 delimited by end-of-file (wanted `MD')


patch cell; it rewrites just the README and avoids the ! lines in code fences:

In [3]:
%%bash
set -euo pipefail
BASE=/content/oncopanel
cat <<'EOF' > "$BASE/README.md"
# oncopanel
Boring, fast, audit-proof panel evaluation at scale (Colab-friendly).

## Quickstart (Colab)
Run these in a new Colab cell:
    pip install -e /content/oncopanel
    oncopanel --help
EOF
echo "README fixed."


README fixed.


In [4]:
# Cell 2 — install oncopanel (editable)
!pip -q install -e /content/oncopanel


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build editable ... error
error: subprocess-exited-with-error

× Getting requirements to build editable did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


**patch**

In [5]:
%%bash
set -euo pipefail
BASE=/content/oncopanel

# Rewrite pyproject with explicit setuptools package discovery
cat > "$BASE/pyproject.toml" <<'TOML'
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "oncopanel"
version = "0.1.0"
description = "Panel-scale context vs codon evaluation (calibration, DCA, bootstrap)"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
  "numpy",
  "pandas",
  "pyarrow",
  "polars>=0.20.0",
  "scikit-learn",
  "typer>=0.12",
  "pandera>=0.18",
]

[project.scripts]
oncopanel = "oncopanel.cli:main"

[tool.setuptools]
include-package-data = true

[tool.setuptools.packages.find]
where = ["."]
include = ["oncopanel*"]
TOML

echo "pyproject.toml fixed."


pyproject.toml fixed.


In [6]:
# Cell — upgrade build tools, then install oncopanel (editable)
!python -m pip -q install --upgrade pip setuptools wheel
!pip -q install -e /content/oncopanel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for oncopanel (pyproject.toml) ... done


In [7]:
# Cell — smoke test CLI + import
!oncopanel --help || python -m oncopanel.cli --help

import oncopanel
print("oncopanel import OK from:", oncopanel.__file__)


Traceback (most recent call last):
  File "/usr/local/bin/oncopanel", line 3, in <module>
    from oncopanel.cli import main
ModuleNotFoundError: No module named 'oncopanel'
/usr/bin/python3: No module named oncopanel.cli
oncopanel import OK from: /content/oncopanel/__init__.py


In [8]:
# Cell — verify/repair CLI module, then smoke test import
from pathlib import Path
BASE = Path("/content/oncopanel")
CLI = BASE / "cli.py"

cli_src = r'''
import typer, yaml, time, pandas as pd
from oncopanel.calibrators.base import IsotonicCalibrator, PlattCalibrator
from oncopanel.metrics.core import per_fold
from oncopanel.receipts.manifest import write_manifest

app = typer.Typer(no_args_is_help=True)
CALIBS = {"isotonic": IsotonicCalibrator, "platt": PlattCalibrator}

@app.command()
def evaluate(
    data_path: str,
    column_map: str = "",
    calibrator: str = "isotonic",
    outdir: str = "./artifacts"
):
    df = pd.read_parquet(data_path) if data_path.endswith(".parquet") else pd.read_csv(data_path)
    # TODO: apply column map & validation (validators/schema.py)
    y = df["label"].to_numpy()
    folds = df["fold"].to_numpy()
    p_ctx = df["score_context"].to_numpy()
    p_cdn = df["score_codon"].to_numpy()

    cal = CALIBS[calibrator]()
    p_ctx_cal, card = cal.fit_predict_oof(p_ctx, y, folds)

    per = per_fold(y, p_ctx_cal, p_cdn, folds)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out = f"{outdir}/per_fold_{ts}.csv"
    import os; os.makedirs(outdir, exist_ok=True)
    per.to_csv(out, index=False)
    write_manifest(outdir, inputs={"data": data_path, "column_map": column_map}, extras={"calibrator": card.__dict__})
    print("wrote:", out)

def main():
    app()
'''

# Ensure package dir exists
assert BASE.is_dir(), f"Package dir not found: {BASE}"

# (Re)write CLI if missing or empty
if (not CLI.exists()) or CLI.stat().st_size == 0:
    CLI.write_text(cli_src.strip() + "\n")
    print("cli.py created.")
else:
    print("cli.py already present.")

# Smoke test import
import importlib, oncopanel
print("oncopanel package:", oncopanel.__file__)
cli_mod = importlib.import_module("oncopanel.cli")
print("oncopanel.cli:", cli_mod.__file__)
print("✅ CLI import OK")


cli.py created.
oncopanel package: /content/oncopanel/__init__.py


ModuleNotFoundError: No module named 'oncopanel.calibrators.base'

In [9]:
# Cell — verify/repair scaffold, then re-test imports
from pathlib import Path
BASE = Path("/content/oncopanel")
assert BASE.is_dir(), f"Package dir not found: {BASE}"

# --- helpers ---
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def ensure_file(p: Path, src: str):
    if not p.exists() or p.stat().st_size == 0:
        p.write_text(src.strip() + "\n")
        print("created:", p)
    else:
        print("ok:", p)

# --- ensure package dirs + __init__.py ---
subdirs = ["", "dataloaders", "validators", "calibrators", "metrics", "bootstrap", "dca", "aggregate", "receipts"]
for d in subdirs:
    dpath = BASE / d if d else BASE
    ensure_dir(dpath)
    initf = dpath / "__init__.py"
    ensure_file(initf, "__all__ = []")

# --- critical modules (create if missing) ---
calib_base = r"""
from dataclasses import dataclass
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

@dataclass
class CalibCard:
    name: str
    params: dict
    monotone: bool
    n_folds: int

class Calibrator:
    name = "base"
    def fit_predict_oof(self, scores, y, folds, groups=None):
        raise NotImplementedError

class IsotonicCalibrator(Calibrator):
    name = "isotonic"
    def fit_predict_oof(self, scores, y, folds, groups=None):
        scores = np.asarray(scores); y = np.asarray(y); folds = np.asarray(folds)
        preds = np.zeros_like(scores, dtype=np.float32)
        uniq = np.unique(folds)
        for f in uniq:
            tr = folds != f; te = folds == f
            iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
            iso.fit(scores[tr], y[tr])
            preds[te] = iso.predict(scores[te])
        return preds, CalibCard(self.name, {"out_of_bounds":"clip"}, True, int(uniq.size))

class PlattCalibrator(Calibrator):
    name = "platt"
    def fit_predict_oof(self, scores, y, folds, groups=None):
        scores = np.asarray(scores); y = np.asarray(y); folds = np.asarray(folds)
        preds = np.zeros_like(scores, dtype=np.float32)
        uniq = np.unique(folds)
        for f in uniq:
            tr = folds != f; te = folds == f
            lr = LogisticRegression(solver="lbfgs", max_iter=200)
            lr.fit(scores[tr].reshape(-1,1), y[tr])
            preds[te] = lr.predict_proba(scores[te].reshape(-1,1))[:,1]
        return preds, CalibCard(self.name, {"solver":"lbfgs"}, False, int(uniq.size))
"""

metrics_core = r"""
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def per_fold(y, p_ctx, p_cdn, fold, weights=None) -> pd.DataFrame:
    rows = []
    y = np.asarray(y); p_ctx = np.asarray(p_ctx); p_cdn = np.asarray(p_cdn); fold = np.asarray(fold)
    for f in np.unique(fold):
        m = (fold == f)
        if m.sum() < 3:
            continue
        w = None if weights is None else np.asarray(weights)[m]
        y_f, c_f, d_f = y[m], p_ctx[m], p_cdn[m]
        rows.append({
            "fold": int(f),
            "auroc_ctx": roc_auc_score(y_f, c_f, sample_weight=w),
            "auroc_cdn": roc_auc_score(y_f, d_f, sample_weight=w),
            "ap_ctx": average_precision_score(y_f, c_f, sample_weight=w),
            "ap_cdn": average_precision_score(y_f, d_f, sample_weight=w),
            "brier_ctx": brier_score_loss(y_f, c_f, sample_weight=w),
            "brier_cdn": brier_score_loss(y_f, d_f, sample_weight=w),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["Δauroc"] = df["auroc_ctx"] - df["auroc_cdn"]
    df["Δap"]    = df["ap_ctx"]    - df["ap_cdn"]
    df["Δbrier"] = df["brier_ctx"] - df["brier_cdn"]
    return df
"""

receipts_manifest = r"""
import json, subprocess, sys, hashlib, pathlib, platform

def _md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for ch in iter(lambda: f.read(1<<20), b''): h.update(ch)
    return h.hexdigest()

def write_manifest(outdir, inputs, extras=None):
    p = pathlib.Path(outdir); p.mkdir(parents=True, exist_ok=True)
    env = subprocess.getoutput(f"{sys.executable} -m pip freeze")
    git = subprocess.getoutput("git rev-parse HEAD || echo 'no-git'")
    man = {"inputs": inputs, "md5": {k: _md5(v) for k,v in inputs.items() if v},
           "python": sys.version, "platform": platform.platform(),
           "pip_freeze": env, "git_sha": git, "extras": extras or {}}
    (p/"manifest.json").write_text(json.dumps(man, indent=2))
"""

validators_schema = r"""
def validate_df(df):
    req = ["variant_id","label","fold","__site_id","score_context","score_codon"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return True
"""

cli_src = r"""
import typer, yaml, time, pandas as pd
from oncopanel.calibrators.base import IsotonicCalibrator, PlattCalibrator
from oncopanel.metrics.core import per_fold
from oncopanel.receipts.manifest import write_manifest

app = typer.Typer(no_args_is_help=True)
CALIBS = {"isotonic": IsotonicCalibrator, "platt": PlattCalibrator}

@app.command()
def evaluate(
    data_path: str,
    column_map: str = "",
    calibrator: str = "isotonic",
    outdir: str = "./artifacts"
):
    df = pd.read_parquet(data_path) if data_path.endswith(".parquet") else pd.read_csv(data_path)
    # TODO: apply column map & validation
    y = df["label"].to_numpy()
    folds = df["fold"].to_numpy()
    p_ctx = df["score_context"].to_numpy()
    p_cdn = df["score_codon"].to_numpy()

    cal = CALIBS[calibrator]()
    p_ctx_cal, card = cal.fit_predict_oof(p_ctx, y, folds)

    per = per_fold(y, p_ctx_cal, p_cdn, folds)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out = f"{outdir}/per_fold_{ts}.csv"
    import os; os.makedirs(outdir, exist_ok=True)
    per.to_csv(out, index=False)
    write_manifest(outdir, inputs={"data": data_path, "column_map": column_map}, extras={"calibrator": card.__dict__})
    print("wrote:", out)

def main():
    app()
"""

ensure_file(BASE/"calibrators"/"base.py", calib_base)
ensure_file(BASE/"metrics"/"core.py",      metrics_core)
ensure_file(BASE/"receipts"/"manifest.py", receipts_manifest)
ensure_file(BASE/"validators"/"schema.py", validators_schema)
ensure_file(BASE/"cli.py",                 cli_src)

# --- try imports now ---
import importlib
print("\nRe-testing imports...")
import oncopanel
print("oncopanel:", oncopanel.__file__)
mod_cal = importlib.import_module("oncopanel.calibrators.base")
print("oncopanel.calibrators.base:", mod_cal.__file__)
mod_cli = importlib.import_module("oncopanel.cli")
print("oncopanel.cli:", mod_cli.__file__)
print("✅ imports OK")


ok: /content/oncopanel/__init__.py
ok: /content/oncopanel/dataloaders/__init__.py
ok: /content/oncopanel/validators/__init__.py
ok: /content/oncopanel/calibrators/__init__.py
ok: /content/oncopanel/metrics/__init__.py
ok: /content/oncopanel/bootstrap/__init__.py
ok: /content/oncopanel/dca/__init__.py
ok: /content/oncopanel/aggregate/__init__.py
ok: /content/oncopanel/receipts/__init__.py
created: /content/oncopanel/calibrators/base.py
created: /content/oncopanel/metrics/core.py
created: /content/oncopanel/receipts/manifest.py
created: /content/oncopanel/validators/schema.py
ok: /content/oncopanel/cli.py

Re-testing imports...
oncopanel: /content/oncopanel/__init__.py
oncopanel.calibrators.base: /content/oncopanel/calibrators/base.py
oncopanel.cli: /content/oncopanel/cli.py
✅ imports OK


In [10]:
# Cell — CLI help (module form)
!python -m oncopanel.cli --help


In [11]:
# Cell — demo data → CLI evaluate → show per-fold deltas
import numpy as np, pandas as pd, os, glob

# 1) Make a small synthetic dataset with a bit of signal
rng = np.random.default_rng(7)
n = 240
folds = rng.integers(0, 5, size=n).astype("int8")
sites = rng.choice(list("ABCDE"), size=n)             # 5 sites
labels = rng.integers(0, 2, size=n).astype("int8")    # 0/1 labels

# codon baseline: weak signal
z_cdn = (labels * 0.6) + rng.normal(0, 1.0, size=n)
p_cdn = 1 / (1 + np.exp(-z_cdn))
# context scores: slightly better calibrated signal
z_ctx = (labels * 0.9) + rng.normal(0, 1.0, size=n)
p_ctx = 1 / (1 + np.exp(-z_ctx))

df = pd.DataFrame({
    "variant_id": [f"v{i}" for i in range(n)],
    "label": labels,
    "fold": folds,
    "__site_id": sites,
    "score_context": p_ctx.astype("float32"),
    "score_codon":   p_cdn.astype("float32"),
})

parquet_path = "/content/demo.parquet"
df.to_parquet(parquet_path)
print("Wrote demo:", parquet_path, "| shape:", df.shape, "| folds:", sorted(df['fold'].unique()))

# 2) Run CLI evaluate
os.makedirs("/content/artifacts", exist_ok=True)
!python -m oncopanel.cli evaluate /content/demo.parquet --outdir /content/artifacts

# 3) Load the newest per-fold CSV and display
paths = sorted(glob.glob("/content/artifacts/per_fold_*.csv"))
assert paths, "No per_fold_* CSV found"
latest = paths[-1]
print("Per-fold metrics:", latest)

out = pd.read_csv(latest)
display(out)


Wrote demo: /content/demo.parquet | shape: (240, 6) | folds: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]


AssertionError: No per_fold_* CSV found

**patch**

In [12]:
# Cell — patch CLI so data_path is a positional argument
from pathlib import Path

cli_src = r'''
import typer, time, pandas as pd
from oncopanel.calibrators.base import IsotonicCalibrator, PlattCalibrator
from oncopanel.metrics.core import per_fold
from oncopanel.receipts.manifest import write_manifest

app = typer.Typer(no_args_is_help=True)
CALIBS = {"isotonic": IsotonicCalibrator, "platt": PlattCalibrator}

@app.command()
def evaluate(
    data_path: str = typer.Argument(..., help="Path to Parquet/CSV with standard columns"),
    column_map: str = typer.Option("", "--column-map", "-m", help="(optional) column map YAML"),
    calibrator: str = typer.Option("isotonic", "--calibrator", "-c", help="isotonic|platt"),
    outdir: str = typer.Option("./artifacts", "--outdir", "-o", help="Output directory")
):
    df = pd.read_parquet(data_path) if data_path.endswith(".parquet") else pd.read_csv(data_path)
    # TODO: apply column map & validation
    y = df["label"].to_numpy()
    folds = df["fold"].to_numpy()
    p_ctx = df["score_context"].to_numpy()
    p_cdn = df["score_codon"].to_numpy()

    cal = CALIBS[calibrator]()
    p_ctx_cal, card = cal.fit_predict_oof(p_ctx, y, folds)

    per = per_fold(y, p_ctx_cal, p_cdn, folds)
    ts = time.strftime("%Y%m%d_%H%M%S")
    import os; os.makedirs(outdir, exist_ok=True)
    out_csv = f"{outdir}/per_fold_{ts}.csv"
    per.to_csv(out_csv, index=False)
    write_manifest(outdir, inputs={"data": data_path, "column_map": column_map}, extras={"calibrator": card.__dict__})
    print("wrote:", out_csv)

def main():
    app()
'''

p = Path("/content/oncopanel/cli.py")
p.write_text(cli_src.strip() + "\n")
print("patched:", p)


patched: /content/oncopanel/cli.py


verify what’s in `/content/oncopanel/cli.py` and confirm it imports:

In [13]:
# Cell — peek at cli.py + confirm import
from pathlib import Path
import importlib

p = Path("/content/oncopanel/cli.py")
print("File:", p)
print("="*70)
print(p.read_text())

cli_mod = importlib.import_module("oncopanel.cli")
print("\n✅ oncopanel.cli imported from:", cli_mod.__file__)


File: /content/oncopanel/cli.py
import typer, time, pandas as pd
from oncopanel.calibrators.base import IsotonicCalibrator, PlattCalibrator
from oncopanel.metrics.core import per_fold
from oncopanel.receipts.manifest import write_manifest

app = typer.Typer(no_args_is_help=True)
CALIBS = {"isotonic": IsotonicCalibrator, "platt": PlattCalibrator}

@app.command()
def evaluate(
    data_path: str = typer.Argument(..., help="Path to Parquet/CSV with standard columns"),
    column_map: str = typer.Option("", "--column-map", "-m", help="(optional) column map YAML"),
    calibrator: str = typer.Option("isotonic", "--calibrator", "-c", help="isotonic|platt"),
    outdir: str = typer.Option("./artifacts", "--outdir", "-o", help="Output directory")
):
    df = pd.read_parquet(data_path) if data_path.endswith(".parquet") else pd.read_csv(data_path)
    # TODO: apply column map & validation
    y = df["label"].to_numpy()
    folds = df["fold"].to_numpy()
    p_ctx = df["score_context"].to_numpy()


execute `evaluate`, show the per-fold table:

In [14]:
# Cell — run evaluate → display per-fold deltas
import os, glob, pandas as pd, subprocess, sys

# Ensure demo parquet exists (recreate if needed)
if not os.path.exists("/content/demo.parquet"):
    import numpy as np
    rng = np.random.default_rng(7)
    n = 240
    folds = rng.integers(0, 5, size=n).astype("int8")
    sites = rng.choice(list("ABCDE"), size=n)
    labels = rng.integers(0, 2, size=n).astype("int8")
    z_cdn = (labels * 0.6) + rng.normal(0, 1.0, size=n)
    p_cdn = 1 / (1 + np.exp(-z_cdn))
    z_ctx = (labels * 0.9) + rng.normal(0, 1.0, size=n)
    p_ctx = 1 / (1 + np.exp(-z_ctx))
    pd.DataFrame({
        "variant_id": [f"v{i}" for i in range(n)],
        "label": labels,
        "fold": folds,
        "__site_id": sites,
        "score_context": p_ctx.astype("float32"),
        "score_codon":   p_cdn.astype("float32"),
    }).to_parquet("/content/demo.parquet")
    print("Rewrote demo: /content/demo.parquet")

# Run the CLI (module form)
res = subprocess.run(
    [sys.executable, "-m", "oncopanel.cli", "evaluate", "/content/demo.parquet", "--outdir", "/content/artifacts"],
    capture_output=True, text=True
)
print(res.stdout or res.stderr)

# Load newest per-fold CSV and display
paths = sorted(glob.glob("/content/artifacts/per_fold_*.csv"))
assert paths, "No per_fold_* CSV found in /content/artifacts"
latest = paths[-1]
print("Per-fold CSV:", latest)

df_out = pd.read_csv(latest)
df_out


AssertionError: No per_fold_* CSV found in /content/artifacts

let’s surface the actual error and see if the CSV got written somewhere unexpected.

In [15]:
# Cell — run CLI visibly, then list artifacts
import os, pandas as pd

# sanity: show demo file + a peek
assert os.path.exists("/content/demo.parquet"), "demo.parquet missing"
print("demo exists. size(bytes):", os.path.getsize("/content/demo.parquet"))
print(pd.read_parquet("/content/demo.parquet").head(3))

# run the CLI in the foreground to see messages/errors
!python -m oncopanel.cli evaluate /content/demo.parquet --outdir /content/artifacts

# list outputs
!ls -la /content/artifacts || true


demo exists. size(bytes): 8030
  variant_id  label  fold __site_id  score_context  score_codon
0         v0      1     4         E       0.836340     0.597411
1         v1      0     3         A       0.642021     0.553409
2         v2      0     3         E       0.744777     0.573645
total 8
drwxr-xr-x 2 root root 4096 Aug 28 18:22 .
drwxr-xr-x 1 root root 4096 Aug 28 18:22 ..


patch `cli.py` so it executes when run as a module:

Quick note on the guard: if __name__ == '__main__': main() means “only run main() when this file is executed as a script/module (python -m oncopanel.cli), not when it’s just imported.” Keeps imports clean; scripts actually run.

In [16]:
# Cell — add __main__ guard so `python -m oncopanel.cli ...` actually runs
from pathlib import Path

p = Path("/content/oncopanel/cli.py")
src = p.read_text()

guard = "\nif __name__ == '__main__':\n    main()\n"
if "if __name__ == '__main__':" not in src:
    p.write_text(src.rstrip() + guard)
    print("✅ Appended __main__ guard.")
else:
    print("ℹ️ __main__ guard already present.")

print("\n--- Tail of cli.py ---")
print("\n".join(p.read_text().splitlines()[-12:]))


✅ Appended __main__ guard.

--- Tail of cli.py ---
    per = per_fold(y, p_ctx_cal, p_cdn, folds)
    ts = time.strftime("%Y%m%d_%H%M%S")
    import os; os.makedirs(outdir, exist_ok=True)
    out_csv = f"{outdir}/per_fold_{ts}.csv"
    per.to_csv(out_csv, index=False)
    write_manifest(outdir, inputs={"data": data_path, "column_map": column_map}, extras={"calibrator": card.__dict__})
    print("wrote:", out_csv)

def main():
    app()
if __name__ == '__main__':
    main()


next cell. It runs the CLI (now that the __main__ guard is in place), lists the outputs, and shows the per-fold table:

In [17]:
# Cell — run evaluate → list outputs → display per-fold deltas
!python -m oncopanel.cli evaluate /content/demo.parquet --outdir /content/artifacts
!ls -lah /content/artifacts

import pandas as pd, glob
paths = sorted(glob.glob("/content/artifacts/per_fold_*.csv"))
assert paths, "No per_fold_* CSV found"
df = pd.read_csv(paths[-1])
df


Usage: python -m oncopanel.cli [OPTIONS] DATA_PATH
Try 'python -m oncopanel.cli --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Got unexpected extra argument (/content/demo.parquet)                        │
╰──────────────────────────────────────────────────────────────────────────────╯
total 8.0K
drwxr-xr-x 2 root root 4.0K Aug 28 18:22 .
drwxr-xr-x 1 root root 4.0K Aug 28 18:22 ..


AssertionError: No per_fold_* CSV found

In [18]:
# Cell — make CLI a simple single-command (no subcommands)
from pathlib import Path

p = Path("/content/oncopanel/cli.py")
src = r'''
import time, os
import pandas as pd
import typer
from oncopanel.calibrators.base import IsotonicCalibrator, PlattCalibrator
from oncopanel.metrics.core import per_fold
from oncopanel.receipts.manifest import write_manifest

CALIBS = {"isotonic": IsotonicCalibrator, "platt": PlattCalibrator}

def _run(
    data_path: str = typer.Argument(..., help="Path to Parquet/CSV with standard columns"),
    column_map: str = typer.Option("", "--column-map", "-m", help="(optional) column map YAML"),
    calibrator: str = typer.Option("isotonic", "--calibrator", "-c", help="isotonic|platt"),
    outdir: str = typer.Option("./artifacts", "--outdir", "-o", help="Output directory")
):
    df = pd.read_parquet(data_path) if data_path.endswith(".parquet") else pd.read_csv(data_path)
    # TODO: apply column map & validation
    y = df["label"].to_numpy()
    folds = df["fold"].to_numpy()
    p_ctx = df["score_context"].to_numpy()
    p_cdn = df["score_codon"].to_numpy()

    cal = CALIBS[calibrator]()
    p_ctx_cal, card = cal.fit_predict_oof(p_ctx, y, folds)

    per = per_fold(y, p_ctx_cal, p_cdn, folds)
    ts = time.strftime("%Y%m%d_%H%M%S")
    os.makedirs(outdir, exist_ok=True)
    out_csv = f"{outdir}/per_fold_{ts}.csv"
    per.to_csv(out_csv, index=False)
    write_manifest(outdir, inputs={"data": data_path, "column_map": column_map}, extras={"calibrator": card.__dict__})
    print("wrote:", out_csv)

def main():
    typer.run(_run)

if __name__ == "__main__":
    main()
'''
p.write_text(src.strip() + "\n")
print("✅ CLI rewritten to single-command form:")
print(p.read_text().splitlines()[0])


✅ CLI rewritten to single-command form:
import time, os


In [19]:
# Cell — run evaluate (root command) → list outputs → display CSV
!python -m oncopanel.cli /content/demo.parquet --outdir /content/artifacts
!ls -lah /content/artifacts

import pandas as pd, glob
paths = sorted(glob.glob("/content/artifacts/per_fold_*.csv"))
assert paths, "No per_fold_* CSV found"
pd.read_csv(paths[-1])


wrote: /content/artifacts/per_fold_20250828_184311.csv
total 28K
drwxr-xr-x 2 root root 4.0K Aug 28 18:43 .
drwxr-xr-x 1 root root 4.0K Aug 28 18:22 ..
-rw-r--r-- 1 root root  15K Aug 28 18:43 manifest.json
-rw-r--r-- 1 root root  967 Aug 28 18:43 per_fold_20250828_184311.csv


,fold,auroc_ctx,auroc_cdn,ap_ctx,ap_cdn,brier_ctx,brier_cdn,Δauroc,Δap,Δbrier
0,0,0.808442,0.735931,0.757655,0.729791,0.202069,0.209067,0.072511,0.027863,-0.006998
1,1,0.736842,0.510526,0.748134,0.562549,0.217029,0.286186,0.226316,0.185585,-0.069158
2,2,0.750299,0.661888,0.697759,0.662064,0.203016,0.231444,0.088411,0.035695,-0.028427
3,3,0.726190,0.662338,0.708377,0.678432,0.205206,0.237233,0.063853,0.029946,-0.032028
4,4,0.753086,0.704938,0.747005,0.636195,0.200897,0.216207,0.048148,0.110810,-0.015311


Fresh data, not old. That CSV came from the synthetic demo.parquet we just cooked and the CLI just wrote it (see the timestamp suffix). If you wanna prove it, run this single cell to show provenance + mtimes:

In [20]:
# Cell — prove it's fresh: manifest + mtimes + MD5s
import json, glob, os, hashlib, pandas as pd, datetime as dt

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1<<20), b""):
            h.update(ch)
    return h.hexdigest()

arts = sorted(glob.glob("/content/artifacts/per_fold_*.csv"))
assert arts, "No per_fold_* CSV found"
latest = arts[-1]
man = "/content/artifacts/manifest.json"
demo = "/content/demo.parquet"

print("Latest CSV:", latest)
print("CSV mtime:", dt.datetime.fromtimestamp(os.path.getmtime(latest)))
print("Manifest mtime:", dt.datetime.fromtimestamp(os.path.getmtime(man)))
print("Demo.parquet mtime:", dt.datetime.fromtimestamp(os.path.getmtime(demo)))

with open(man) as f:
    manifest = json.load(f)
print("\nManifest inputs:", manifest.get("inputs"))
print("Manifest MD5(data):", manifest.get("md5", {}).get("data"))
print("Recomputed MD5(demo):", md5(demo))
print("Calibrator card:", manifest.get("extras", {}).get("calibrator"))

print("\nPer-fold head:")
print(pd.read_csv(latest).head())


Latest CSV: /content/artifacts/per_fold_20250828_184311.csv
CSV mtime: 2025-08-28 18:43:11.193311
Manifest mtime: 2025-08-28 18:43:12.125392
Demo.parquet mtime: 2025-08-28 18:22:10.275362

Manifest inputs: {'data': '/content/demo.parquet', 'column_map': ''}
Manifest MD5(data): 119138d4a6a145177c28c6620e4e0d8f
Recomputed MD5(demo): 119138d4a6a145177c28c6620e4e0d8f
Calibrator card: {'name': 'isotonic', 'params': {'out_of_bounds': 'clip'}, 'monotone': True, 'n_folds': 5}

Per-fold head:
   fold  auroc_ctx  auroc_cdn    ap_ctx    ap_cdn  brier_ctx  brier_cdn  \
0     0   0.808442   0.735931  0.757655  0.729791   0.202069   0.209067   
1     1   0.736842   0.510526  0.748134  0.562549   0.217029   0.286186   
2     2   0.750299   0.661888  0.697759  0.662064   0.203016   0.231444   
3     3   0.726190   0.662338  0.708377  0.678432   0.205206   0.237233   
4     4   0.753086   0.704938  0.747005  0.636195   0.200897   0.216207   

     Δauroc       Δap    Δbrier  
0  0.072511  0.027863 -0.0

In [23]:
# Cell — add strict schema + validator (Pandera) to oncopanel/validators/schema.py
from pathlib import Path

schema_src = r"""
import pandas as pd
import pandera as pa
from pandera import Check
from typing import Iterable, Tuple, Optional

def get_standard_schema() -> pa.DataFrameSchema:
    return pa.DataFrameSchema(
        {
            # required
            "variant_id":   pa.Column(pa.String, checks=[Check.str_length(min_value=1)]),
            "label":        pa.Column(pa.Int8,   checks=[Check.isin([0,1])]),
            "fold":         pa.Column(pa.Int16),
            "__site_id":    pa.Column(pa.String, checks=[Check.str_length(min_value=1)]),
            "score_context":pa.Column(pa.Float32, checks=[Check.ge(0.0), Check.le(1.0)]),
            "score_codon":  pa.Column(pa.Float32, checks=[Check.ge(0.0), Check.le(1.0)]),

            # optional (allowed to be missing)
            "patient_id":   pa.Column(pa.String,   required=False, nullable=True),
            "gene":         pa.Column(pa.String,   required=False, nullable=True),
            "cohort":       pa.Column(pa.String,   required=False, nullable=True),
            "split_tag":    pa.Column(pa.String,   required=False, nullable=True),
            "timestamp":    pa.Column(pa.DateTime, required=False, nullable=True),
            "sample_weight":pa.Column(pa.Float32,  required=False, nullable=True, checks=[Check.ge(0.0)]),
            "group_id":     pa.Column(pa.String,   required=False, nullable=True),
        },
        coerce=True,
        strict=False,  # allow extra columns, we won't rely on them
    )

def _assert_unique(df: pd.DataFrame, keys: Iterable[str]) -> None:
    if not keys:
        return
    dups = df.duplicated(list(keys), keep=False)
    if bool(dups.any()):
        bad = df.loc[dups, list(keys)].head(10).to_dict(orient="records")
        raise ValueError(f"Expected unique rows by {list(keys)}, found duplicates. Examples: {bad}")

def _assert_min_sites(df: pd.DataFrame, col: str="__site_id", min_sites: int=2) -> None:
    n = df[col].dropna().nunique()
    if n < min_sites:
        raise ValueError(f"Not enough distinct {col}: got {n}, need >= {min_sites}")

def _assert_fold_labels(df: pd.DataFrame, col: str="fold") -> None:
    if not pd.api.types.is_integer_dtype(df[col].dtype):
        raise ValueError(f"fold must be integer dtype, got {df[col].dtype}")
    # sanity: at least 2 distinct folds
    if df[col].nunique() < 2:
        raise ValueError("need at least 2 distinct folds")

def validate_df(
    df: pd.DataFrame,
    unique_keys: Tuple[str, ...] = ("variant_id",),
    min_sites: int = 2,
) -> pd.DataFrame:
    """
    # Validate and coerce to the standard schema.
    # Extra columns are allowed; required columns are enforced.
    # Also enforces uniqueness and minimal site diversity.
"""
    schema = get_standard_schema()
    df2 = schema.validate(df, lazy=False)  # coerce dtypes, check ranges
    _assert_unique(df2, unique_keys)
    _assert_min_sites(df2, "__site_id", min_sites=min_sites)
    _assert_fold_labels(df2, "fold")
    return df2
"""

p = Path("/content/oncopanel/validators/schema.py")
p.write_text(schema_src.strip() + "\n")
print("✅ wrote validator:", p)


✅ wrote validator: /content/oncopanel/validators/schema.py


In [25]:
# Cell — fix & write dataloader with YAML column-map + validator
from pathlib import Path

src = r'''
# oncopanel/dataloaders/io.py
from __future__ import annotations

from typing import Dict, Iterable, Optional
from pathlib import Path
import pandas as pd
import yaml

from oncopanel.validators.schema import validate_df

# ---- basic readers ----
def _read_table(path: str | Path) -> pd.DataFrame:
    """Read a Parquet/CSV table; prefer Parquet."""
    p = str(path)
    if p.endswith((".parquet", ".pq")):
        return pd.read_parquet(p)
    if p.endswith(".csv"):
        return pd.read_csv(p)
    try:
        return pd.read_parquet(p)
    except Exception:
        return pd.read_csv(p)

# ---- column mapping helpers ----
def _pick_first_present(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    for c in (candidates or []):
        if c in df.columns:
            return c
    return None

def _apply_column_map(df: pd.DataFrame, cfg: Dict) -> pd.DataFrame:
    """Rename source columns to standard names using configs/column_map.yaml."""
    out = df.copy()
    scores = cfg.get("scores", {})
    ctx_aliases = scores.get("context", [])
    cdn_aliases = scores.get("codon", [])

    # only rename if standard name not already present
    if "score_context" not in out.columns:
        src = _pick_first_present(out, ctx_aliases)
        if src:
            out = out.rename(columns={src: "score_context"})
    if "score_codon" not in out.columns:
        src = _pick_first_present(out, cdn_aliases)
        if src:
            out = out.rename(columns={src: "score_codon"})
    return out

# ---- public API ----
def load_table(
    path: str | Path,
    column_map_yaml: str | Path | None = None,
    enforce_schema: bool = True,
) -> pd.DataFrame:
    """
    Load a table (Parquet/CSV), apply YAML column-map, then validate to the standard schema.
    Required columns after mapping: variant_id, label, fold, __site_id, score_context, score_codon
    """
    df = _read_table(path)

    if column_map_yaml:
        cfg = yaml.safe_load(Path(column_map_yaml).read_text())
        df = _apply_column_map(df, cfg)

    if enforce_schema:
        df = validate_df(df)

    return df
'''

dst = Path("/content/oncopanel/dataloaders/io.py")
dst.write_text(src.strip() + "\n")
print("✅ wrote dataloader:", dst)


✅ wrote dataloader: /content/oncopanel/dataloaders/io.py


wire the dataloader + validator into the CLI and make every run write to a timestamped folder so filenames are always unique.

In [26]:
# Cell — patch CLI to use dataloader + timestamped run dirs
from pathlib import Path

p = Path("/content/oncopanel/cli.py")
src = r'''
import time, os
import pandas as pd
import typer
from oncopanel.dataloaders.io import load_table
from oncopanel.calibrators.base import IsotonicCalibrator, PlattCalibrator
from oncopanel.metrics.core import per_fold
from oncopanel.receipts.manifest import write_manifest

CALIBS = {"isotonic": IsotonicCalibrator, "platt": PlattCalibrator}

def _run(
    data_path: str = typer.Argument(..., help="Path to Parquet/CSV with (or mappable to) standard columns"),
    column_map: str = typer.Option("", "--column-map", "-m", help="YAML mapping if your columns use other names"),
    calibrator: str = typer.Option("isotonic", "--calibrator", "-c", help="isotonic|platt"),
    outdir: str = typer.Option("./artifacts", "--outdir", "-o", help="Base output directory")
):
    # 1) Load + validate (column-map applied if provided)
    df = load_table(data_path, column_map_yaml=column_map or None, enforce_schema=True)

    # 2) Pull arrays
    y      = df["label"].to_numpy()
    folds  = df["fold"].to_numpy()
    p_ctx  = df["score_context"].to_numpy()
    p_cdn  = df["score_codon"].to_numpy()

    # 3) OOF calibration
    cal = CALIBS[calibrator]()
    p_ctx_cal, card = cal.fit_predict_oof(p_ctx, y, folds)

    # 4) Per-fold metrics (ctx vs codon)
    per = per_fold(y, p_ctx_cal, p_cdn, folds)

    # 5) Timestamped run folder + outputs
    run_id = time.strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(outdir, f"RUN_{run_id}")
    os.makedirs(run_dir, exist_ok=True)

    out_csv = os.path.join(run_dir, f"per_fold_{run_id}.csv")
    per.to_csv(out_csv, index=False)

    write_manifest(
        run_dir,
        inputs={"data": data_path, "column_map": column_map},
        extras={"calibrator": card.__dict__, "run_id": run_id}
    )

    print(f"wrote: {out_csv}")
    print(f"manifest: {os.path.join(run_dir, 'manifest.json')}")

def main():
    typer.run(_run)

if __name__ == "__main__":
    main()
'''
p.write_text(src.strip() + "\n")
print("✅ CLI patched to use dataloader + timestamped run dirs:", p)


✅ CLI patched to use dataloader + timestamped run dirs: /content/oncopanel/cli.py
